# 📽️ PLAYWRIGHT - Data Preprocessing

This notebook processes raw scripts to create training data for beat segmentation:
1. Parse scene boundaries (INT./EXT. markers)
2. Identify beat boundaries within scenes
3. Extract features for each segment
4. Create labeled training dataset

---
**Runtime:** ~20-40 minutes

## Setup

In [ ]:
!pip install pandas numpy tqdm -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import json
import pickle
from typing import List, Dict, Optional
from dataclasses import dataclass
from tqdm import tqdm
import pandas as pd
import numpy as np

# Data paths
DATA_PATH = "/content/drive/MyDrive/playwright_data"
CORNELL_PATH = f"{DATA_PATH}/cornell"
IMSDB_PATH = f"{DATA_PATH}/imsdb"
OUTPUT_PATH = f"{DATA_PATH}/processed"

os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"📁 Data path: {DATA_PATH}")

In [ ]:
# Load IMSDB scripts
df_scripts = pd.read_parquet(f"{IMSDB_PATH}/imsdb_scripts.parquet")
print(f"✅ Loaded {len(df_scripts)} scripts")

## 1. Script Parsing Utilities

In [ ]:
@dataclass
class ScriptElement:
    """Represents a parsed element from a screenplay."""
    element_type: str  # SCENE_HEADING, ACTION, CHARACTER, DIALOGUE, PARENTHETICAL, TRANSITION
    text: str
    line_number: int
    raw_text: str

In [ ]:
class ScreenplayParser:
    """Parser for standard screenplay format."""

    SCENE_HEADING_PATTERN = re.compile(
        r'^(INT\.|EXT\.|INT/EXT\.|I/E\.)[\s]+(.+?)(?:\s*[-–—]\s*(.+))?$',
        re.IGNORECASE
    )

    CHARACTER_PATTERN = re.compile(r'^([A-Z][A-Z\s\.\'\'\-]+)(\s*\(.*\))?$')

    PARENTHETICAL_PATTERN = re.compile(r'^\s*\(.*\)\s*$')

    TRANSITION_PATTERN = re.compile(
        r'^(FADE IN:|FADE OUT:|CUT TO:|DISSOLVE TO:|SMASH CUT:|MATCH CUT:|FADE TO BLACK|THE END).*$',
        re.IGNORECASE
    )

    def __init__(self):
        self.elements = []
        self.scenes = []

    def parse(self, script_text: str) -> List[ScriptElement]:
        """Parse a screenplay into structured elements."""
        lines = script_text.split('\n')
        self.elements = []

        current_character = None

        for i, line in enumerate(lines):
            raw_line = line
            line = line.strip()

            if not line:
                continue

            element = self._classify_line(line, i, raw_line, current_character)
            if element:
                self.elements.append(element)
                if element.element_type == 'CHARACTER':
                    current_character = element.text
                elif element.element_type in ['SCENE_HEADING', 'ACTION']:
                    current_character = None

        return self.elements

    def _classify_line(
        self,
        line: str,
        line_num: int,
        raw_line: str,
        current_character: Optional[str]
    ) -> Optional[ScriptElement]:
        """Classify a single line of the screenplay."""

        # Scene heading
        if self.SCENE_HEADING_PATTERN.match(line):
            return ScriptElement('SCENE_HEADING', line, line_num, raw_line)

        # Transition
        if self.TRANSITION_PATTERN.match(line):
            return ScriptElement('TRANSITION', line, line_num, raw_line)

        # Parenthetical
        if self.PARENTHETICAL_PATTERN.match(line):
            return ScriptElement('PARENTHETICAL', line, line_num, raw_line)

        # Character name
        if self.CHARACTER_PATTERN.match(line) and len(line) < 50:
            char_match = self.CHARACTER_PATTERN.match(line)
            char_name = char_match.group(1).strip()
            if char_name not in ['THE', 'A', 'AN', 'AND', 'BUT', 'OR']:
                return ScriptElement('CHARACTER', char_name, line_num, raw_line)

        # Dialogue (if we have a current character)
        if current_character and not line.isupper():
            return ScriptElement('DIALOGUE', line, line_num, raw_line)

        # Default to action
        return ScriptElement('ACTION', line, line_num, raw_line)

    def extract_scenes(self) -> List[Dict]:
        """Extract scenes from parsed elements."""
        scenes = []
        current_scene = None

        for element in self.elements:
            if element.element_type == 'SCENE_HEADING':
                if current_scene:
                    scenes.append(current_scene)

                match = self.SCENE_HEADING_PATTERN.match(element.text)
                current_scene = {
                    'scene_id': len(scenes),
                    'heading': element.text,
                    'location_type': match.group(1) if match else '',
                    'location': match.group(2) if match else '',
                    'time_of_day': match.group(3) if match else '',
                    'start_line': element.line_number,
                    'elements': []
                }
            elif current_scene:
                current_scene['elements'].append({
                    'type': element.element_type,
                    'text': element.text,
                    'line': element.line_number
                })

        if current_scene:
            scenes.append(current_scene)

        return scenes

## 2. Beat Segmentation Logic

In [ ]:
class BeatSegmenter:
    """Segments scenes into visual beats."""

    BEAT_TRIGGERS = [
        'suddenly', 'then', 'but', 'meanwhile', 'later',
        'moments later', 'beat', 'pause', 'silence',
        'cut to', 'angle on', 'close on', 'wide on',
        'pov', 'reverse', 'two shot', 'over shoulder'
    ]

    MOOD_KEYWORDS = {
        'tense': ['gun', 'knife', 'blood', 'scream', 'fear', 'danger', 'threat', 'dark'],
        'romantic': ['kiss', 'love', 'embrace', 'tender', 'gentle', 'heart', 'passion'],
        'action': ['run', 'fight', 'chase', 'explode', 'crash', 'punch', 'kick', 'shoot'],
        'mysterious': ['shadow', 'hidden', 'secret', 'strange', 'unknown', 'eerie'],
        'comedic': ['laugh', 'joke', 'funny', 'smile', 'grin', 'absurd'],
        'dramatic': ['cry', 'tears', 'angry', 'shout', 'confront', 'reveal'],
        'serene': ['calm', 'peaceful', 'quiet', 'still', 'gentle', 'soft'],
        'melancholic': ['sad', 'alone', 'lonely', 'grief', 'loss', 'memory']
    }

    CAMERA_INDICATORS = {
        'close-up': ['close on', 'close-up', 'cu on', 'tight on', 'eyes', 'face', 'hand'],
        'wide shot': ['wide', 'establishing', 'aerial', 'landscape', 'vista'],
        'medium shot': ['medium', 'waist', 'two shot', 'group'],
        'pov shot': ['pov', 'point of view', 'subjective', 'through eyes'],
        'over-the-shoulder': ['over shoulder', 'ots', 'behind'],
        'tracking shot': ['follows', 'tracking', 'dolly', 'moving with'],
        'low angle': ['low angle', 'looking up', 'from below'],
        'high angle': ['high angle', 'looking down', 'from above', 'bird']
    }

    def segment_scene(self, scene: Dict) -> List[Dict]:
        """Segment a scene into beats."""
        elements = scene.get('elements', [])
        if not elements:
            return []

        beats = []
        current_beat_elements = []
        beat_id = 0

        for i, element in enumerate(elements):
            should_split = self._should_split_beat(element, current_beat_elements, elements, i)

            if should_split and current_beat_elements:
                beat = self._create_beat(
                    beat_id,
                    current_beat_elements,
                    scene['scene_id'],
                    scene
                )
                beats.append(beat)
                beat_id += 1
                current_beat_elements = []

            current_beat_elements.append(element)

        # Don't forget the last beat
        if current_beat_elements:
            beat = self._create_beat(
                beat_id,
                current_beat_elements,
                scene['scene_id'],
                scene
            )
            beats.append(beat)

        return beats

    def _should_split_beat(
        self,
        element: Dict,
        current_elements: List[Dict],
        all_elements: List[Dict],
        index: int
    ) -> bool:
        """Determine if a new beat should start."""
        if not current_elements:
            return False

        text_lower = element['text'].lower()

        # Check for beat trigger words
        for trigger in self.BEAT_TRIGGERS:
            if trigger in text_lower:
                return True

        # New character speaking after several lines of dialogue
        if element['type'] == 'CHARACTER':
            dialogue_count = sum(1 for e in current_elements if e['type'] == 'DIALOGUE')
            if dialogue_count >= 4:
                return True

        # Long action block
        if element['type'] == 'ACTION':
            action_count = sum(1 for e in current_elements if e['type'] == 'ACTION')
            if action_count >= 3:
                return True

        # Beat getting too long
        total_text = ' '.join(e['text'] for e in current_elements)
        if len(total_text) > 500:
            return True

        return False

    def _create_beat(
        self,
        beat_id: int,
        elements: List[Dict],
        scene_id: int,
        scene: Dict
    ) -> Dict:
        """Create a beat dictionary from elements."""
        full_text = ' '.join(e['text'] for e in elements)

        # Extract characters
        characters = list(set(
            e['text'] for e in elements if e['type'] == 'CHARACTER'
        ))

        # Get action and dialogue text
        action_text = ' '.join(
            e['text'] for e in elements if e['type'] == 'ACTION'
        )
        dialogue_text = ' '.join(
            e['text'] for e in elements if e['type'] == 'DIALOGUE'
        )

        # Detect mood and camera
        mood = self._detect_mood(full_text)
        camera = self._suggest_camera(full_text, elements)

        # Generate visual description
        visual_desc = self._generate_visual_description(
            action_text,
            characters,
            scene,
            elements
        )

        return {
            'beat_id': beat_id,
            'scene_id': scene_id,
            'start_line': elements[0]['line'] if elements else 0,
            'end_line': elements[-1]['line'] if elements else 0,
            'full_text': full_text,
            'action_text': action_text,
            'dialogue_text': dialogue_text,
            'characters': characters,
            'mood': mood,
            'suggested_camera': camera,
            'visual_description': visual_desc,
            'element_count': len(elements),
            'has_dialogue': any(e['type'] == 'DIALOGUE' for e in elements),
            'has_action': any(e['type'] == 'ACTION' for e in elements)
        }

    def _detect_mood(self, text: str) -> str:
        """Detect the mood of a beat."""
        text_lower = text.lower()
        mood_scores = {}

        for mood, keywords in self.MOOD_KEYWORDS.items():
            score = sum(1 for kw in keywords if kw in text_lower)
            if score > 0:
                mood_scores[mood] = score

        if mood_scores:
            return max(mood_scores, key=mood_scores.get)
        return 'neutral'

    def _suggest_camera(self, text: str, elements: List[Dict]) -> str:
        """Suggest camera angle based on content."""
        text_lower = text.lower()

        for camera, indicators in self.CAMERA_INDICATORS.items():
            if any(ind in text_lower for ind in indicators):
                return camera

        # Default based on character count
        dialogue_count = sum(1 for e in elements if e['type'] == 'DIALOGUE')
        char_count = len(set(e['text'] for e in elements if e['type'] == 'CHARACTER'))

        if char_count == 1 and dialogue_count > 0:
            return 'close-up'
        elif char_count == 2:
            return 'over-the-shoulder'
        elif char_count > 2:
            return 'wide shot'

        return 'medium shot'

    def _generate_visual_description(
        self,
        action_text: str,
        characters: List[str],
        scene: Dict,
        elements: List[Dict]
    ) -> str:
        """Generate a visual description for the beat."""
        parts = []

        # Add location info
        location = scene.get('location', '')
        time = scene.get('time_of_day', '')
        loc_type = scene.get('location_type', '')

        if location:
            setting = f"{loc_type} {location}".strip()
            if time:
                setting += f" - {time}"
            parts.append(setting)

        # Add characters
        if characters:
            if len(characters) == 1:
                parts.append(f"{characters[0]} in frame")
            else:
                parts.append(f"{', '.join(characters[:3])} in scene")

        # Add action summary
        if action_text:
            action_summary = action_text[:200].strip()
            if action_summary:
                parts.append(action_summary)

        return '. '.join(parts) if parts else "Scene moment"

## 3. Process Scripts and Create Training Data

In [ ]:
def process_script_to_beats(script_text: str, script_id: str) -> List[Dict]:
    """Process a full script into beats."""
    parser = ScreenplayParser()
    segmenter = BeatSegmenter()

    parser.parse(script_text)
    scenes = parser.extract_scenes()

    all_beats = []
    global_beat_id = 0

    for scene in scenes:
        scene_beats = segmenter.segment_scene(scene)
        for beat in scene_beats:
            beat['global_beat_id'] = global_beat_id
            beat['script_id'] = script_id
            beat['scene_heading'] = scene.get('heading', '')
            global_beat_id += 1
        all_beats.extend(scene_beats)

    return all_beats

In [ ]:
# Process all scripts
print("🔄 Processing scripts into beats...")

all_beats_data = []
errors = []

for idx, row in tqdm(df_scripts.iterrows(), total=len(df_scripts), desc="Processing"):
    try:
        beats = process_script_to_beats(row['text'], row['script_name'])
        all_beats_data.extend(beats)
    except Exception as e:
        errors.append({'script': row['script_name'], 'error': str(e)})

print(f"\n✅ Processed {len(df_scripts) - len(errors)} scripts")
print(f"❌ Errors: {len(errors)}")
print(f"📊 Total beats extracted: {len(all_beats_data):,}")

In [ ]:
# Convert to DataFrame and save
df_beats = pd.DataFrame(all_beats_data)

# Save beats
df_beats.to_parquet(f"{OUTPUT_PATH}/training_beats.parquet", index=False)
print(f"💾 Saved {len(df_beats):,} beats to {OUTPUT_PATH}/training_beats.parquet")

# Preview
df_beats.head()

## 4. Create Sequence Labeling Dataset

In [ ]:
def create_sequence_labels(script_text: str, script_id: str) -> List[Dict]:
    """Create BIO-tagged sequence labels for training."""
    parser = ScreenplayParser()
    segmenter = BeatSegmenter()

    parser.parse(script_text)
    scenes = parser.extract_scenes()

    labeled_sequences = []

    for scene in scenes:
        scene_beats = segmenter.segment_scene(scene)

        for beat_idx, beat in enumerate(scene_beats):
            # Get elements for this beat based on line numbers
            beat_elements = [
                e for e in scene['elements']
                if beat['start_line'] <= e['line'] <= beat['end_line']
            ]

            for elem_idx, elem in enumerate(beat_elements):
                # BIO tagging: B = beginning of beat, I = inside beat
                if elem_idx == 0:
                    label = 'B-BEAT'
                else:
                    label = 'I-BEAT'

                labeled_sequences.append({
                    'script_id': script_id,
                    'text': elem['text'],
                    'element_type': elem['type'],
                    'label': label,
                    'beat_id': beat_idx,
                    'scene_id': scene['scene_id'],
                    'mood': beat['mood'],
                    'camera': beat['suggested_camera']
                })

    return labeled_sequences

In [ ]:
# Create sequence labels (use subset for faster processing)
print("🔄 Creating sequence labels...")

all_sequences = []
num_scripts = min(100, len(df_scripts))  # Process up to 100 scripts

for idx, row in tqdm(df_scripts.head(num_scripts).iterrows(), total=num_scripts, desc="Labeling"):
    try:
        sequences = create_sequence_labels(row['text'], row['script_name'])
        all_sequences.extend(sequences)
    except Exception as e:
        pass

print(f"\n✅ Total labeled sequences: {len(all_sequences):,}")

In [ ]:
# Save sequence labels
df_sequences = pd.DataFrame(all_sequences)
df_sequences.to_parquet(f"{OUTPUT_PATH}/sequence_labels.parquet", index=False)
print(f"💾 Saved {len(df_sequences):,} sequences to {OUTPUT_PATH}/sequence_labels.parquet")

# Show label distribution
print("\n📊 Label Distribution:")
print(df_sequences['label'].value_counts())

In [ ]:
# Show mood distribution
print("\n🎭 Mood Distribution:")
print(df_beats['mood'].value_counts())

In [ ]:
# Show camera distribution
print("\n📷 Camera Angle Distribution:")
print(df_beats['suggested_camera'].value_counts())

## Summary

In [ ]:
print("="*50)
print("📊 DATA PREPROCESSING COMPLETE")
print("="*50)
print(f"\n📁 Output saved to: {OUTPUT_PATH}")
print(f"\n📈 Dataset Statistics:")
print(f"  • Training Beats: {len(df_beats):,}")
print(f"  • Sequence Labels: {len(df_sequences):,}")
print(f"  • Unique Moods: {df_beats['mood'].nunique()}")
print(f"  • Unique Camera Angles: {df_beats['suggested_camera'].nunique()}")
print(f"\n✅ Ready for model training (Notebook 03)")

---
## Next Steps

Training data created:
- `training_beats.parquet` - Beat-level data with features
- `sequence_labels.parquet` - BIO-tagged sequences for training

**Proceed to `03_model_training.ipynb`** to train the segmentation model.